# Smart Meter Electricity Consumption Data Pipeline

## Bronze Layer - Raw Data Ingestion

### Objective

This notebook performs the following tasks:

- Read meter_readings.csv
- Read household_info.csv
- Explore the datasets
- Validate schema
- Save data as Delta Tables

Technology Used:
- PySpark
- Delta Lake
- Databricks

In [0]:
meter_df = spark.read.format("csv") \
.option("header","true") \
.option("inferSchema","true") \
.load("/Volumes/workspace/default/smart_meter_data/meter_readings (1).csv")

In [0]:
display(meter_df)

meter_id,household_id,timestamp,units_consumed
M001,H001,2026-04-01T00:00:00.000Z,0.41
M002,H002,2026-04-01T00:00:00.000Z,0.1
M003,H003,2026-04-01T00:00:00.000Z,0.44
M004,H004,2026-04-01T00:00:00.000Z,0.23
M005,H005,2026-04-01T00:00:00.000Z,6.93
M006,H006,2026-04-01T00:00:00.000Z,0.8
M007,H007,2026-04-01T00:00:00.000Z,0.47
M008,H008,2026-04-01T00:00:00.000Z,0.2
M009,H009,2026-04-01T00:00:00.000Z,0.25
M010,H010,2026-04-01T00:00:00.000Z,0.25


In [0]:
meter_df.printSchema()

root
 |-- meter_id: string (nullable = true)
 |-- household_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- units_consumed: double (nullable = true)



In [0]:
meter_df.count()

1400

In [0]:
len(meter_df.columns)

4

In [0]:
meter_df.columns

['meter_id', 'household_id', 'timestamp', 'units_consumed']

In [0]:
meter_df.describe().show()

+-------+--------+------------+------------------+
|summary|meter_id|household_id|    units_consumed|
+-------+--------+------------+------------------+
|  count|    1400|        1400|              1380|
|   mean|    NULL|        NULL|2.2665217391304395|
| stddev|    NULL|        NULL|2.7529690879815636|
|    min|    M001|        H001|               0.0|
|    max|    M010|        H010|             38.37|
+-------+--------+------------+------------------+



In [0]:
household_df = spark.read.format("csv") \
.option("header","true") \
.option("inferSchema","true") \
.load("/Volumes/workspace/default/smart_meter_data/household_info (1).csv")

In [0]:
display(household_df)

household_id,city,house_type,avg_daily_consumption
H001,Jaipur,Independent,8
H002,Kolkata,Independent,10
H003,Jaipur,Villa,12
H004,Jaipur,Apartment,15
H005,Kolkata,Apartment,7
H006,Mumbai,Studio,9
H007,Pune,Studio,11
H008,Delhi,Studio,14
H009,Pune,Independent,6
H010,Hyderabad,Independent,13


In [0]:
household_df.printSchema()

root
 |-- household_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- house_type: string (nullable = true)
 |-- avg_daily_consumption: integer (nullable = true)



In [0]:
household_df.count()

10

In [0]:
len(household_df.columns)

4

In [0]:
household_df.columns

['household_id', 'city', 'house_type', 'avg_daily_consumption']

In [0]:
household_df.describe().show()

+-------+------------+-----+----------+---------------------+
|summary|household_id| city|house_type|avg_daily_consumption|
+-------+------------+-----+----------+---------------------+
|  count|          10|   10|        10|                   10|
|   mean|        NULL| NULL|      NULL|                 10.5|
| stddev|        NULL| NULL|      NULL|   3.0276503540974917|
|    min|        H001|Delhi| Apartment|                    6|
|    max|        H010| Pune|     Villa|                   15|
+-------+------------+-----+----------+---------------------+



In [0]:
meter_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_meter_readings")

In [0]:
household_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_household_info")

In [0]:
%sql
SHOW TABLES IN workspace.default;

database,tableName,isTemporary
default,bronze_household_info,false
default,bronze_meter_readings,false
default,customer-incremental,false
default,customer-master,false
default,customer_incremental,false
default,customer_master_delta,false
default,inc,false
default,master,false
default,sample-superstore,false
default,sample_superstore,false


In [0]:
%sql
SELECT * FROM workspace.default.bronze_meter_readings LIMIT 10;

meter_id,household_id,timestamp,units_consumed
M001,H001,2026-04-01T00:00:00.000Z,0.41
M002,H002,2026-04-01T00:00:00.000Z,0.1
M003,H003,2026-04-01T00:00:00.000Z,0.44
M004,H004,2026-04-01T00:00:00.000Z,0.23
M005,H005,2026-04-01T00:00:00.000Z,6.93
M006,H006,2026-04-01T00:00:00.000Z,0.8
M007,H007,2026-04-01T00:00:00.000Z,0.47
M008,H008,2026-04-01T00:00:00.000Z,0.2
M009,H009,2026-04-01T00:00:00.000Z,0.25
M010,H010,2026-04-01T00:00:00.000Z,0.25


In [0]:
%sql
SELECT * FROM workspace.default.bronze_household_info LIMIT 10;

household_id,city,house_type,avg_daily_consumption
H001,Jaipur,Independent,8
H002,Kolkata,Independent,10
H003,Jaipur,Villa,12
H004,Jaipur,Apartment,15
H005,Kolkata,Apartment,7
H006,Mumbai,Studio,9
H007,Pune,Studio,11
H008,Delhi,Studio,14
H009,Pune,Independent,6
H010,Hyderabad,Independent,13


In [0]:
meter_df.filter(meter_df.units_consumed.isNull()).count()

20

In [0]:
meter_df.dropDuplicates(["meter_id", "timestamp"]).count()

1400

In [0]:
meter_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_meter_readings")

In [0]:
household_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_household_info")

In [0]:
%sql
SHOW TABLES IN workspace.default;

database,tableName,isTemporary
default,bronze_household_info,false
default,bronze_meter_readings,false
default,customer-incremental,false
default,customer-master,false
default,customer_incremental,false
default,customer_master_delta,false
default,inc,false
default,master,false
default,sample-superstore,false
default,sample_superstore,false


In [0]:
spark.sql("SHOW TABLES IN workspace.default").show()

+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
| default|bronze_household_...|      false|
| default|bronze_meter_read...|      false|
| default|customer-incremental|      false|
| default|     customer-master|      false|
| default|customer_incremental|      false|
| default|customer_master_d...|      false|
| default|                 inc|      false|
| default|              master|      false|
| default|   sample-superstore|      false|
| default|   sample_superstore|      false|
| default|              source|      false|
| default|    superstore_delta|      false|
+--------+--------------------+-----------+



In [0]:
%sql
SELECT * FROM workspace.default.bronze_meter_readings LIMIT 10;

meter_id,household_id,timestamp,units_consumed
M001,H001,2026-04-01T00:00:00.000Z,0.41
M002,H002,2026-04-01T00:00:00.000Z,0.1
M003,H003,2026-04-01T00:00:00.000Z,0.44
M004,H004,2026-04-01T00:00:00.000Z,0.23
M005,H005,2026-04-01T00:00:00.000Z,6.93
M006,H006,2026-04-01T00:00:00.000Z,0.8
M007,H007,2026-04-01T00:00:00.000Z,0.47
M008,H008,2026-04-01T00:00:00.000Z,0.2
M009,H009,2026-04-01T00:00:00.000Z,0.25
M010,H010,2026-04-01T00:00:00.000Z,0.25


In [0]:
%sql
SELECT * FROM workspace.default.bronze_household_info LIMIT 10;

household_id,city,house_type,avg_daily_consumption
H001,Jaipur,Independent,8
H002,Kolkata,Independent,10
H003,Jaipur,Villa,12
H004,Jaipur,Apartment,15
H005,Kolkata,Apartment,7
H006,Mumbai,Studio,9
H007,Pune,Studio,11
H008,Delhi,Studio,14
H009,Pune,Independent,6
H010,Hyderabad,Independent,13


In [0]:
from pyspark.sql.functions import current_timestamp, to_date, col
meter_df = meter_df.withColumn("_ingestion_time", current_timestamp())

household_df = household_df.withColumn("_ingestion_time", current_timestamp())

In [0]:
meter_df = meter_df.withColumn(
    "ingestion_date",
    to_date(col("_ingestion_time"))
)

In [0]:
%sql
SHOW TABLES IN workspace.default;

database,tableName,isTemporary
default,bronze_household_info,false
default,bronze_meter_readings,false
default,customer-incremental,false
default,customer-master,false
default,customer_incremental,false
default,customer_master_delta,false
default,inc,false
default,master,false
default,sample-superstore,false
default,sample_superstore,false


In [0]:
%sql
DESCRIBE TABLE EXTENDED workspace.default.bronze_meter_readings;

col_name,data_type,comment
meter_id,string,null
household_id,string,null
timestamp,timestamp,null
units_consumed,double,null
_ingestion_time,timestamp,null
,,
# Delta Statistics Columns,,
Column Names,"household_id, timestamp, units_consumed, _ingestion_time, meter_id",
Column Selection Method,first-32,
,,


In [0]:
%sql
DROP TABLE IF EXISTS workspace.default.bronze_meter_readings;


In [0]:
%sql
DROP TABLE IF EXISTS workspace.default.bronze_household_info;

In [0]:
from pyspark.sql.functions import current_timestamp

meter_df = meter_df.withColumn(
    "_ingestion_time",
    current_timestamp()
)

household_df = household_df.withColumn(
    "_ingestion_time",
    current_timestamp()
)

In [0]:
meter_df.printSchema()

root
 |-- meter_id: string (nullable = true)
 |-- household_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- units_consumed: double (nullable = true)
 |-- _ingestion_time: timestamp (nullable = false)



In [0]:
%sql
DROP TABLE IF EXISTS workspace.default.bronze_meter_readings;

In [0]:
%sql
DROP TABLE IF EXISTS workspace.default.bronze_household_info;

In [0]:
meter_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_meter_readings")

In [0]:
household_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_household_info")

In [0]:
spark.sql("DESCRIBE TABLE workspace.default.bronze_meter_readings").show(truncate=False)

+---------------+---------+-------+
|col_name       |data_type|comment|
+---------------+---------+-------+
|meter_id       |string   |NULL   |
|household_id   |string   |NULL   |
|timestamp      |timestamp|NULL   |
|units_consumed |double   |NULL   |
|_ingestion_time|timestamp|NULL   |
|ingestion_date |date     |NULL   |
+---------------+---------+-------+



In [0]:
%sql
SELECT * FROM workspace.default.bronze_meter_readings LIMIT 10;

meter_id,household_id,timestamp,units_consumed,_ingestion_time
M001,H001,2026-04-01T00:00:00.000Z,0.41,2026-07-15T13:23:43.674Z
M002,H002,2026-04-01T00:00:00.000Z,0.1,2026-07-15T13:23:43.674Z
M003,H003,2026-04-01T00:00:00.000Z,0.44,2026-07-15T13:23:43.674Z
M004,H004,2026-04-01T00:00:00.000Z,0.23,2026-07-15T13:23:43.674Z
M005,H005,2026-04-01T00:00:00.000Z,6.93,2026-07-15T13:23:43.674Z
M006,H006,2026-04-01T00:00:00.000Z,0.8,2026-07-15T13:23:43.674Z
M007,H007,2026-04-01T00:00:00.000Z,0.47,2026-07-15T13:23:43.674Z
M008,H008,2026-04-01T00:00:00.000Z,0.2,2026-07-15T13:23:43.674Z
M009,H009,2026-04-01T00:00:00.000Z,0.25,2026-07-15T13:23:43.674Z
M010,H010,2026-04-01T00:00:00.000Z,0.25,2026-07-15T13:23:43.674Z


In [0]:
%sql
SELECT * FROM workspace.default.bronze_household_info limit 10;

household_id,city,house_type,avg_daily_consumption,_ingestion_time
H001,Jaipur,Independent,8,2026-07-15T13:23:55.587Z
H002,Kolkata,Independent,10,2026-07-15T13:23:55.587Z
H003,Jaipur,Villa,12,2026-07-15T13:23:55.587Z
H004,Jaipur,Apartment,15,2026-07-15T13:23:55.587Z
H005,Kolkata,Apartment,7,2026-07-15T13:23:55.587Z
H006,Mumbai,Studio,9,2026-07-15T13:23:55.587Z
H007,Pune,Studio,11,2026-07-15T13:23:55.587Z
H008,Delhi,Studio,14,2026-07-15T13:23:55.587Z
H009,Pune,Independent,6,2026-07-15T13:23:55.587Z
H010,Hyderabad,Independent,13,2026-07-15T13:23:55.587Z


In [0]:
display(spark.table("workspace.default.bronze_meter_readings"))

meter_id,household_id,timestamp,units_consumed,_ingestion_time,ingestion_date
M001,H001,2026-04-01T00:00:00.000Z,0.41,2026-07-19T08:13:53.569Z,2026-07-19
M002,H002,2026-04-01T00:00:00.000Z,0.1,2026-07-19T08:13:53.569Z,2026-07-19
M003,H003,2026-04-01T00:00:00.000Z,0.44,2026-07-19T08:13:53.569Z,2026-07-19
M004,H004,2026-04-01T00:00:00.000Z,0.23,2026-07-19T08:13:53.569Z,2026-07-19
M005,H005,2026-04-01T00:00:00.000Z,6.93,2026-07-19T08:13:53.569Z,2026-07-19
M006,H006,2026-04-01T00:00:00.000Z,0.8,2026-07-19T08:13:53.569Z,2026-07-19
M007,H007,2026-04-01T00:00:00.000Z,0.47,2026-07-19T08:13:53.569Z,2026-07-19
M008,H008,2026-04-01T00:00:00.000Z,0.2,2026-07-19T08:13:53.569Z,2026-07-19
M009,H009,2026-04-01T00:00:00.000Z,0.25,2026-07-19T08:13:53.569Z,2026-07-19
M010,H010,2026-04-01T00:00:00.000Z,0.25,2026-07-19T08:13:53.569Z,2026-07-19
